# Minutas verificables: Ministral 3B + JSON restringido + herramientas

**Entrega 2 · Generative Artificial Intelligence.** Ejecuta en Google Colab con `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)` y después `Entorno de ejecución → Ejecutar todo`. Cuando aparezca el selector, sube `Fine Tuning.zip`. El notebook genera los tres brazos **prompt directo**, **JSON restringido** y **JSON restringido + herramientas** con el *mismo modelo y las mismas reuniones*. No entrena ningún modelo ni usa la rúbrica durante inferencia.

Este notebook está configurado para la evaluación final en las diez reuniones de test. No necesitas editar ninguna celda. Las respuestas se guardan tras cada llamada para poder reanudar una ejecución interrumpida. El ZIP de resultados se descarga desde la última celda.

**Diseño de la intervención:** el primer paso genera candidatos sujetos a un esquema JSON. Python consulta las intervenciones citadas, sus vecinas y resuelve fechas explícitas en relación con la fecha de la reunión. Ministral recibe esas salidas de herramientas y emite la minuta corregida, también restringida por esquema. Son herramientas locales deterministas, orquestadas por el sistema; no afirmamos que el modelo las haya invocado mediante function calling autónomo.

In [ ]:
# Instalación y comprobación. Ollama se ejecuta en la VM de Colab.
import subprocess, time, shutil
if not shutil.which('nvidia-smi'):
    raise RuntimeError('Esta sesión de Colab no tiene GPU. Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4 y vuelve a ejecutar desde aquí.')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('Colab no detecta una GPU NVIDIA activa. Selecciona GPU T4 y reconecta el entorno.')
print('GPU detectada:',gpu.stdout.strip())
if not shutil.which('zstd'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], check=True)
if not shutil.which('ollama'):
    subprocess.run(['bash', '-o', 'pipefail', '-c', 'curl -fsSL https://ollama.com/install.sh | sh'], check=True)
if not shutil.which('ollama'):
    raise RuntimeError('La instalación terminó sin dejar el ejecutable ollama en PATH')
%pip -q install requests jsonschema

import requests
from pathlib import Path

ollama_log = open('/content/ollama_serve.log', 'a')
server = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        if requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError('Ollama no inició; revisa /content/ollama_serve.log')

MODEL = 'ministral-3:3b-instruct-2512-q4_K_M'
print('GPU:')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=False)
print('Descargando modelo; solo se hace una vez por sesión de Colab...')
subprocess.run(['ollama', 'pull', MODEL], check=True)
print('Modelo:', MODEL)

In [ ]:
from google.colab import files
import io, json, zipfile
from collections import defaultdict
from pathlib import Path

ROOT = Path('/content/minutas_d2')
ROOT.mkdir(exist_ok=True)
DATA = ROOT / 'data'
DATA.mkdir(exist_ok=True)
print('Sube Fine Tuning.zip:')
uploaded = files.upload()
zipname = next((k for k in uploaded if k.lower().endswith('.zip')), None)
if not zipname:
    raise ValueError('Se necesita el archivo Fine Tuning.zip')

with zipfile.ZipFile(io.BytesIO(uploaded[zipname])) as z:
    for info in z.infolist():
        if info.is_dir() or not info.filename.endswith('.json'):
            continue
        p = Path(info.filename)
        split = next((s for s in ('train', 'validation', 'test') if s in p.parts), None)
        if split is None:
            continue
        target = DATA / split / p.name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(z.read(info))

def load_split(split):
    transcripts, gold = {}, {}
    for path in sorted((DATA / split).glob('*.json')):
        item = json.loads(path.read_text(encoding='utf-8'))
        bucket = transcripts if 'utterances' in item else gold
        mid = item['meeting_id']
        if mid in bucket: raise ValueError(f'ID duplicado en {split}: {mid}')
        bucket[mid] = item
    if set(transcripts) != set(gold):
        raise ValueError(f'Pares incompletos en {split}: {set(transcripts)^set(gold)}')
    for mid, tr in transcripts.items():
        ids = {u['id'] for u in tr['utterances']}
        for item in gold[mid]['decisions'] + gold[mid]['tasks']:
            if item['evidence_id'] not in ids:
                raise ValueError(f'Evidencia inválida en rúbrica {mid}: {item}')
    return transcripts, gold

DATASETS = {s: load_split(s) for s in ('train', 'validation', 'test')}
for split, (tr, gold) in DATASETS.items():
    print(split, len(tr), 'reuniones,', sum(map(lambda x: len(x['utterances']), tr.values())), 'intervenciones')

In [ ]:
# Evaluación final: diez reuniones de test. DEMO_ONLY=True limita la ejecución a M01 para el video.
RUN_FINAL_TEST = True
DEMO_ONLY = False
SPLIT = 'test' if RUN_FINAL_TEST else 'validation'
LIMIT = 1 if DEMO_ONLY else (None if RUN_FINAL_TEST else 2)
TRANSCRIPTS, GOLD = DATASETS[SPLIT]
MEETING_IDS = sorted(TRANSCRIPTS)[:LIMIT]

RESULTS = ROOT / 'results' / SPLIT
RESULTS.mkdir(parents=True, exist_ok=True)
PARAMS = {'temperature': 0, 'seed': 42, 'num_ctx': 12288, 'num_predict': 2400}
print('Conjunto:', SPLIT, '| reuniones:', MEETING_IDS, '| demo:', DEMO_ONLY)
print('NOTA: DEMO_ONLY debe quedar en False para reproducir la tabla oficial de 10 reuniones.')

In [ ]:
import re, datetime as dt
from jsonschema import Draft202012Validator

STATES = ['final', 'superseded', 'rejected', 'pending']

def nullable_string(): return {'type': ['string', 'null']}

def make_schema(tr):
    evidence_ids = [u['id'] for u in tr['utterances']]
    decision = {
      'type': 'object', 'additionalProperties': False,
      'required': ['id','topic','statement','state','condition','supersedes','evidence_id'],
      'properties': {
        'id': {'type':'string'}, 'topic': {'type':'string'}, 'statement': {'type':'string'},
        'state': {'type':'string','enum':STATES}, 'condition': nullable_string(),
        'supersedes': nullable_string(), 'evidence_id': {'type':'string','enum':evidence_ids}
      }
    }
    task = {
      'type': 'object', 'additionalProperties': False,
      'required': ['id','description','assignee','deadline','condition','state','evidence_id'],
      'properties': {
        'id': {'type':'string'}, 'description': {'type':'string'},
        'assignee': nullable_string(),
        'deadline': nullable_string(), 'condition': nullable_string(),
        'state': {'type':'string','enum':STATES},
        'evidence_id': {'type':'string','enum':evidence_ids}
      }
    }
    return {'type':'object','additionalProperties':False,'required':['decisions','tasks'],
            'properties': {'decisions':{'type':'array','items':decision},
                           'tasks':{'type':'array','items':task}}}

SYSTEM = """Genera minutas verificables en español de una reunión COMPLETA. Devuelve exclusivamente un JSON con decisions y tasks.
Cada decisión tiene id, topic, statement, state, condition, supersedes, evidence_id.
Cada tarea tiene id, description, assignee, deadline, condition, state, evidence_id.
Estados: final si hay acuerdo que nunca fue revertido; superseded si una propuesta fue reemplazada (conserva ambos elementos y enlázalos); rejected solo con descarte explícito; pending si no existe cierre observable.
Una propuesta sin consenso no es final. Registra decisiones y tareas que aparecen explícitamente, incluso cuando son reemplazadas después. Distingue una elección de una obligación asignada. Solo asigna a una persona cuando la transcripción le encargue la acción; menciones de terceros no constituyen asignaciones.
Usa como evidencia un ID real cuya intervención respalde el campo; no inventes valores. Fechas absolutas ISO YYYY-MM-DD o YYYY-MM-DDTHH:MM según la fecha de la reunión; si no puedes resolverla, null. Campos no expresados: null. IDs de decisiones d01... y tareas t01... separados. No uses el texto literal "null"."""

def transcript_text(tr):
    head = {k: tr.get(k) for k in ('meeting_id','meeting_date','title','participants')}
    lines = [json.dumps(head, ensure_ascii=False), 'INTERVENCIONES EN ORDEN:']
    lines += [f"{u['id']} | {u.get('timestamp','')} | {u['speaker']}: {u['text']}" for u in tr['utterances']]
    return '\n'.join(lines)

def inference(messages, schema=None):
    body = {'model':MODEL, 'messages':messages, 'stream':False, 'options':PARAMS}
    if schema is not None: body['format'] = schema
    t0 = time.monotonic()
    res = requests.post('http://127.0.0.1:11434/api/chat', json=body, timeout=900)
    res.raise_for_status()
    out = res.json()
    raw = out['message'].get('content','')
    try:
        parsed = json.loads(raw)
        if not isinstance(parsed, dict): parsed = None
    except (json.JSONDecodeError, TypeError): parsed = None
    return {'raw':raw, 'parsed':parsed, 'seconds':round(time.monotonic()-t0,2),
            'eval_count':out.get('eval_count'), 'prompt_eval_count':out.get('prompt_eval_count'),
            'schema_valid':bool(parsed is not None and schema is not None and Draft202012Validator(schema).is_valid(parsed))}

def save_json(path, value):
    path.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding='utf-8')

def run_cached(mid, arm, fn):
    path = RESULTS / f'{mid}_{arm}.json'
    if path.exists():
        print('  reutilizado:', path.name)
        return json.loads(path.read_text(encoding='utf-8'))
    result = fn()
    save_json(path, result)
    print(' ', arm, f"{result['seconds']}s", 'JSON:', result['parsed'] is not None)
    return result

### Herramientas locales de verificación

`inspect_evidence` devuelve la intervención citada y contexto próximo; `resolve_deadline` calcula fechas explícitas respecto de `meeting_date`, sin consultar la rúbrica. La segunda llamada a Ministral recibe sus informes para corregir atribuciones, fechas y citas. Una herramienta puede comprobar que existe un ID, pero **la implicación semántica** de una cita sigue siendo una decisión del modelo y se medirá contra el gold.

In [ ]:
MONTHS = {'enero':1,'febrero':2,'marzo':3,'abril':4,'mayo':5,'junio':6,
          'julio':7,'agosto':8,'septiembre':9,'setiembre':9,'octubre':10,
          'noviembre':11,'diciembre':12}
WEEKDAYS = {'lunes':0,'martes':1,'miércoles':2,'miercoles':2,'jueves':3,
            'viernes':4,'sábado':5,'sabado':5,'domingo':6}
WORD_HOURS = {'una':1,'dos':2,'tres':3,'cuatro':4,'cinco':5,'seis':6,
              'siete':7,'ocho':8,'nueve':9,'diez':10,'once':11,'doce':12}

def inspect_evidence(tr, evidence_id):
    utterances = tr['utterances']
    pos = next((i for i,u in enumerate(utterances) if u['id']==evidence_id),None)
    if pos is None: return {'evidence_id':evidence_id,'exists':False}
    return {'evidence_id':evidence_id, 'exists':True,
            'cited':utterances[pos],
            'nearby':[{'id':u['id'],'speaker':u['speaker'],'text':u['text']}
                      for u in utterances[max(0,pos-1):min(len(utterances),pos+2)]],
            'participants':sorted({u['speaker'] for u in utterances})}

def resolve_deadline(tr, evidence_id, draft_value):
    evidence = inspect_evidence(tr,evidence_id)
    if not evidence['exists']:
        return {'evidence_id':evidence_id,'suggested':None,'reason':'ID no existe'}
    base = dt.date.fromisoformat(tr['meeting_date'])
    s = evidence['cited']['text'].lower()
    day = None
    if re.search(r'\bhoy\b',s): day = base
    elif re.search(r'\bpasado mañana\b',s): day = base+dt.timedelta(days=2)
    elif re.search(r'\bmañana\b',s) and not re.search(r'\b(?:de la|por la) mañana\b',s):
        day = base+dt.timedelta(days=1)
    else:
        monthmatch = re.search(r'\b(\d{1,2})\s+de\s+('+'|'.join(MONTHS)+r')\b',s)
        weekdaymatch = re.search(r'\b('+'|'.join(WEEKDAYS)+r')\s+(\d{1,2})\b',s)
        try:
            if monthmatch:
                d, m = int(monthmatch[1]), MONTHS[monthmatch[2]]
                day = dt.date(base.year,m,d)
                if day < base: day = dt.date(base.year+1,m,d)
            elif weekdaymatch:
                d=int(weekdaymatch[2]); day=dt.date(base.year,base.month,d)
                if day < base: day=dt.date(base.year+(base.month==12),(base.month%12)+1,d)
            else:
                wd = next((v for k,v in WEEKDAYS.items() if re.search(r'\b'+k+r'\b',s)),None)
                if wd is not None:
                    delta=(wd-base.weekday())%7
                    day=base+dt.timedelta(days=delta or 7)
        except ValueError:
            day=None
    tm = re.search(r'\b(?:a las|antes de las|a la|antes de la)\s+(\d{1,2})(?::(\d{2}))?\b',s)
    hour = int(tm[1]) if tm else None
    minute = int(tm[2] or 0) if tm else 0
    if hour is None:
        wm = re.search(r'\b(?:a las|antes de las|a la|antes de la)\s+('+'|'.join(WORD_HOURS)+r')\b',s)
        if wm: hour = WORD_HOURS[wm[1]]
    time_ambiguous = hour is not None and hour <= 12 and not re.search(r'\b(?:de la mañana|por la mañana|tarde|noche|mediodía)\b',s)
    if hour is not None and re.search(r'\b(?:tarde|noche)\b',s) and hour <= 11: hour += 12
    if time_ambiguous: hour = None
    if hour is not None and not (0<=hour<24): hour=None
    suggestion = day.isoformat() if day else None
    if suggestion and hour is not None: suggestion += f'T{hour:02d}:{minute:02d}'
    return {'evidence_id':evidence_id,'source_text':evidence['cited']['text'],
            'draft':draft_value, 'suggested':suggestion,
            'reason':('hora ambigua: solo día calculado' if time_ambiguous and suggestion else
                      'fecha explícita calculada' if suggestion else
                      'expresión relativa/ambigua: revisar contexto completo, no forzar fecha')}

def tool_bundle(tr, parsed):
    reports=[]
    for kind in ('decisions','tasks'):
        for item in parsed.get(kind,[]) if isinstance(parsed.get(kind),list) else []:
            if not isinstance(item,dict): continue
            report={'kind':kind,'id':item.get('id'),
                    'evidence':inspect_evidence(tr,item.get('evidence_id'))}
            if kind=='tasks':
                report['assignee_named_in_meeting'] = (item.get('assignee') is None or
                    any(str(item.get('assignee')).lower() in u['text'].lower() or
                        item.get('assignee') == u['speaker'] for u in tr['utterances']))
                if item.get('deadline') is not None:
                    report['deadline_check']=resolve_deadline(tr,item.get('evidence_id'),item.get('deadline'))
            reports.append(report)
    return reports

def revision_cues(tr):
    # Evidencia recuperada de la transcripción; nunca de la rúbrica.
    pattern=r'\b(reemplaz\w*|descart\w*|rechaz\w*|pendiente|aprob\w*|qued\w* a cargo)\b'
    return [{'id':u['id'],'speaker':u['speaker'],'text':u['text']}
            for u in tr['utterances'] if re.search(pattern,u['text'].lower())]

def apply_safe_checks(tr, result):
    # Solo invariantes mecánicas: no se borran errores semánticos para maquillar la evaluación.
    obj=result.get('parsed')
    if not isinstance(obj,dict): return result
    cleaned=json.loads(json.dumps(obj))
    ids={u['id'] for u in tr['utterances']}
    actions=[]
    for kind in ('decisions','tasks'):
        valid=[]
        for item in cleaned.get(kind,[]):
            if not isinstance(item,dict) or item.get('evidence_id') not in ids:
                actions.append('descartado: evidence_id inexistente'); continue
            if kind=='tasks' and item.get('deadline') is not None:
                if not re.fullmatch(r'\d{4}-\d{2}-\d{2}(?:T\d{2}:\d{2}(?::\d{2})?)?',str(item['deadline'])):
                    item['deadline']=None; actions.append('fecha no ISO a null')
                else:
                    try:
                        if 'T' in item['deadline']:
                            item['deadline']=dt.datetime.fromisoformat(item['deadline']).isoformat(timespec='minutes')
                        else: item['deadline']=dt.date.fromisoformat(item['deadline']).isoformat()
                    except ValueError:
                        item['deadline']=None; actions.append('fecha imposible a null')
            valid.append(item)
        cleaned[kind]=valid
    result['parsed']=cleaned
    result['mechanical_actions']=actions
    return result

def run_one(mid):
    tr=TRANSCRIPTS[mid]
    msgs=[{'role':'system','content':SYSTEM},
          {'role':'user','content':transcript_text(tr)}]
    schema=make_schema(tr)
    baseline=run_cached(mid,'baseline',lambda:inference(msgs))
    structured=run_cached(mid,'structured',lambda:inference(msgs,schema))
    def combined_call():
        draft=structured['parsed'] if isinstance(structured['parsed'],dict) else {'decisions':[],'tasks':[]}
        reports=tool_bundle(tr,draft)
        review='BORRADOR (puede contener errores):\n'+json.dumps(draft,ensure_ascii=False)
        review+='\nSALIDAS DE HERRAMIENTAS (no incluyen rúbrica):\n'+json.dumps(reports,ensure_ascii=False)
        review+='\nINDICIOS DE CAMBIOS LOCALIZADOS POR HERRAMIENTA:\n'+json.dumps(revision_cues(tr),ensure_ascii=False)
        review+='\nRevisa todas las intervenciones. En cada reemplazo explícito incluye DOS decisiones: la opción anterior con state=superseded y la nueva con state=final y supersedes apuntando al ID anterior. Reemplazado no significa rejected. Conserva propuestas expresamente descartadas con state=rejected, y asuntos sin acuerdo con state=pending. Revisa todas las tareas confirmadas, incluidas las que luego se completaron. Corrige responsables, fechas y evidencia. Las herramientas verifican existencia y sugieren fechas; NO demuestran por sí solas que la cita entraña cada campo. No atribuyas horas a "hoy" si no se menciona una hora. Devuelve solo la minuta corregida.'
        out=inference(msgs+[{'role':'assistant','content':json.dumps(draft,ensure_ascii=False)},
                            {'role':'user','content':review}],schema)
        out['tool_reports']=reports
        return apply_safe_checks(tr,out)
    # Nueva versión del prompt: no reutilizar respuestas combinadas anteriores.
    combined=run_cached(mid,'combined_v2',combined_call)
    # Recupera la respuesta original para reaplicar los chequeos si se corrigió el código.
    if combined.get('raw'):
        combined['parsed']=json.loads(combined['raw'])
        combined=apply_safe_checks(tr,combined)
        save_json(RESULTS/f'{mid}_combined_v2.json',combined)
    return {'baseline':baseline,'structured':structured,'combined':combined}

In [ ]:
# Evaluación de modelos: se guardan respuestas inmediatamente después de cada llamada.
RUNS={}
for i,mid in enumerate(MEETING_IDS,1):
    print(f'[{i}/{len(MEETING_IDS)}] {mid}')
    try:
        RUNS[mid]=run_one(mid)
    except Exception as exc:
        print('ERROR en',mid,repr(exc),'| reinicia esta celda para reanudar.')
        raise
print('Inferencia terminada.')

### Medición sobre la misma rúbrica

Emparejamiento automático por similitud de contenido y, en segundo término, el ID de evidencia. Se informa precisión, recall y F1 para decisiones/tareas; exactitud del estado, responsable y fecha en elementos emparejados; precisión de los IDs citados; JSON válido y tiempo. Esta aproximación automática debe acompañarse de una auditoría manual de los errores relevantes: **no** equivale a medir si cada afirmación queda semánticamente demostrada por la cita.

In [ ]:
from difflib import SequenceMatcher
import unicodedata
from collections import Counter

def norm(s):
    s=unicodedata.normalize('NFKD',str(s or '').lower())
    s=''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]+',' ',s).strip()

def recover_baseline(raw):
    # Recuperación SOLO para evaluar contenido: raw JSON/schema siguen reportados aparte.
    cleaned=re.sub(r'^\s*```(?:json)?\s*|\s*```\s*$','',raw.strip(),flags=re.I)
    try: obj=json.loads(cleaned)
    except json.JSONDecodeError: return None
    if not isinstance(obj,dict): return None
    return {'decisions':obj.get('decisions',obj.get('decisiones',[])),
            'tasks':obj.get('tasks',obj.get('tareas',[]))}

def comparable_date(value):
    if value is None: return None
    if not isinstance(value,str):return value
    try:
        if re.fullmatch(r'\d{4}-\d{2}-\d{2}',value): return dt.date.fromisoformat(value).isoformat()
        return dt.datetime.fromisoformat(value).isoformat(timespec='minutes')
    except ValueError: return value

def content_score(a,b,kind):
    field='statement' if kind=='decisions' else 'description'
    # La cláusula que anuncia un reemplazo no representa la decisión principal.
    main=re.split(r'\b(?:reemplaz\w*|sustituy\w*)\b',str(a.get(field,'')),maxsplit=1,flags=re.I)[0]
    x,y=norm(main),norm(b.get(field))
    seq=SequenceMatcher(None,x,y).ratio()
    sx,sy=set(x.split()),set(y.split())
    bow=2*len(sx&sy)/(len(sx)+len(sy)) if sx or sy else 0
    gold_coverage=len(sx&sy)/len(sy) if sy else 0
    main=max(seq,bow,gold_coverage)
    topic=SequenceMatcher(None,norm(a.get('topic')),norm(b.get('topic'))).ratio() if kind=='decisions' else 0
    evid=int(a.get('evidence_id')==b.get('evidence_id'))
    return (0.70*main+0.15*topic+0.15*evid) if kind=='decisions' else (0.80*main+0.20*evid)

def match_items(pred,gold,kind):
    candidates=[]
    for i,p in enumerate(pred):
        if not isinstance(p,dict):continue
        for j,g in enumerate(gold):
            score=content_score(p,g,kind)
            if score>=0.55: candidates.append((score,i,j))
    used_p,used_g=set(),set(); pairs=[]
    for score,i,j in sorted(candidates,reverse=True):
        if i not in used_p and j not in used_g:
            used_p.add(i);used_g.add(j);pairs.append((pred[i],gold[j],score))
    return pairs

def evaluate_arm(arm):
    totals=Counter()
    per=[]
    for mid in MEETING_IDS:
        out=RUNS[mid][arm]; obj=out.get('parsed')
        tr=TRANSCRIPTS[mid]; gold=GOLD[mid]
        totals['meetings']+=1
        totals['seconds']+=out['seconds']
        totals['json_valid']+=int(isinstance(obj,dict))
        schema=make_schema(tr)
        totals['schema_valid']+=int(isinstance(obj,dict) and Draft202012Validator(schema).is_valid(obj))
        p=(recover_baseline(out['raw']) if arm=='baseline' else obj)
        p=p if isinstance(p,dict) else {}
        totals['content_recovered']+=int(bool(p))
        for kind in ('decisions','tasks'):
            predicted=p.get(kind,[])
            predicted=predicted if isinstance(predicted,list) else []
            expected=gold[kind]
            pairs=match_items(predicted,expected,kind)
            totals[kind+'_pred']+=len(predicted)
            totals[kind+'_gold']+=len(expected)
            totals[kind+'_tp']+=len(pairs)
            valid_ids={u['id'] for u in tr['utterances']}
            totals['cites']+=sum(isinstance(x,dict) for x in predicted)
            totals['cites_existing']+=sum(isinstance(x,dict) and x.get('evidence_id') in valid_ids for x in predicted)
            for x,y,score in pairs:
                totals['matched']+=1
                totals['state_right']+=int(x.get('state')==y.get('state'))
                totals['evidence_right']+=int(x.get('evidence_id')==y.get('evidence_id'))
                if kind=='tasks':
                    totals['tasks_matched']+=1
                    totals['assignee_right']+=int(x.get('assignee')==y.get('assignee'))
                    totals['deadline_right']+=int(comparable_date(x.get('deadline'))==comparable_date(y.get('deadline')))
        per.append({'meeting_id':mid,'seconds':out['seconds'],
                    'decisions_pred':len(p.get('decisions',[])) if isinstance(p.get('decisions',[]),list) else 0,
                    'tasks_pred':len(p.get('tasks',[])) if isinstance(p.get('tasks',[]),list) else 0})
    result={'arm':arm,'meetings':totals['meetings'],'seconds_total':round(totals['seconds'],1),
            'json_valid':f"{totals['json_valid']}/{totals['meetings']}",
            'schema_valid':f"{totals['schema_valid']}/{totals['meetings']}",
            'content_recovered':f"{totals['content_recovered']}/{totals['meetings']}",
            'state_accuracy_matched':round(totals['state_right']/totals['matched'],3) if totals['matched'] else None,
            'assignee_accuracy_matched':round(totals['assignee_right']/totals['tasks_matched'],3) if totals['tasks_matched'] else None,
            'deadline_accuracy_matched':round(totals['deadline_right']/totals['tasks_matched'],3) if totals['tasks_matched'] else None,
            'evidence_exact_matched':round(totals['evidence_right']/totals['matched'],3) if totals['matched'] else None,
            'evidence_id_exists':f"{totals['cites_existing']}/{totals['cites']}",
            'per_meeting':per,
            'note':'Emparejamiento aproximado automático; revisar errores manualmente.'}
    for kind in ('decisions','tasks'):
        n=totals[kind+'_tp']; pred=totals[kind+'_pred']; expected=totals[kind+'_gold']
        prec=n/pred if pred else (1.0 if not expected else 0.0)
        recall=n/expected if expected else 1.0
        f1=2*prec*recall/(prec+recall) if prec+recall else 0.0
        result[kind]={'matched':n,'predicted':pred,'gold':expected,
                      'precision':round(prec,3),'recall':round(recall,3),'f1':round(f1,3)}
    return result

SUMMARY=[evaluate_arm(arm) for arm in ('baseline','structured','combined')]
save_json(RESULTS/'summary.json',SUMMARY)
for row in SUMMARY:
    print(row['arm'], '| D F1',row['decisions']['f1'],'| T F1',row['tasks']['f1'],
          '| states',row['state_accuracy_matched'], '| owners',row['assignee_accuracy_matched'],
          '| dates',row['deadline_accuracy_matched'],'| evidence',row['evidence_exact_matched'],
          '| schema',row['schema_valid'],'|',row['seconds_total'],'s')
print('IMPORTANTE: compara números del mismo split; no presentes piloto de validation como test final.')

In [ ]:
# Inspección visible para la demostración. M01 se usa por orden, no por ser favorable.
mid=MEETING_IDS[0]
print('REUNIÓN',mid)
print(transcript_text(TRANSCRIPTS[mid]))
print('\n'+'='*80+'\nRÚBRICA (solo evaluación):\n',json.dumps(GOLD[mid],ensure_ascii=False,indent=2))
for arm in ('baseline','structured','combined'):
    print('\n'+'='*80+'\n'+arm.upper())
    print(RUNS[mid][arm]['raw'])
    if arm=='combined':
        print('\nHerramientas consultadas:',len(RUNS[mid][arm].get('tool_reports',[])))

In [ ]:
# Revisión de límites: busca discrepancias de la solución combinada en TEST.
if RUN_FINAL_TEST:
    failures=[]
    for mid in MEETING_IDS:
        obj=RUNS[mid]['combined'].get('parsed') or {}
        for kind in ('decisions','tasks'):
            pred=obj.get(kind,[]) if isinstance(obj.get(kind,[]),list) else []
            gold=GOLD[mid][kind]
            pairs=match_items(pred,gold,kind)
            if len(pairs)<len(gold) or len(pairs)<len(pred):
                failures.append((mid,kind,'omisión o elemento extra',len(pairs),len(pred),len(gold)))
            for p,g,score in pairs:
                for field in (('state','evidence_id') if kind=='decisions' else ('state','evidence_id','assignee','deadline')):
                    if p.get(field)!=g.get(field):
                        failures.append((mid,kind,field,p.get(field),g.get(field),g.get('evidence_id')))
    print('Primeras discrepancias; compruébalas leyendo la transcripción antes de reportar un fallo:')
    for item in failures[:30]: print(item)
else:
    print('Ejecuta RUN_FINAL_TEST = True para buscar el fallo real de test.')

In [ ]:
# Descarga resultados y herramientas de reproducción.
import shutil
archive=shutil.make_archive(str(ROOT/f'minutas_resultados_{SPLIT}'),'zip',root_dir=RESULTS)
print('Archivo:',archive)
files.download(archive)